# CMVP quickstart

Extract a statistically validated backbone from the Zachary karate club
network: fit a configuration-model null, compute p-values for common
neighbors, and threshold at `alpha=0.05` with FDR correction.

In [1]:
import networkx as nx
from cmvp import CMVP

G = nx.karate_club_graph()
for _, _, d in G.edges(data=True):
    d.clear()  # drop edge weights -> use the plain (unweighted) configuration model
print(G)

Graph named "Zachary's Karate Club" with 34 nodes and 78 edges


In [2]:
cmvp = CMVP(G, seed=42)
cmvp.fit_configuration_model(model='auto', method='fixed-point', max_iter=1000)

Auto-selected model: cm_exp (directed=False, weighted=False)
Fitting null model with NEMtropy (model=cm_exp, method=fixed-point)...


Parsed 48 iterations from NEMtropy output
Final |f(x)|: 4.605691e+00
Final diff: 7.034258e-09


In [3]:
# Compute p-values (common neighbors vs. configuration-model null)
cmvp.validate_projection(test='poisson')

# Threshold to a backbone: alpha=0.05, Benjamini-Hochberg FDR correction
cmvp.filter_backbone(alpha=0.05, correction='fdr')

Computing observed common_neighbors similarities...
Found 332 candidate pairs for similarity test
Computing expected common_neighbors (analytical formula)...
Computing p-values under null model (right-tail (similarity), test=poisson)...
Computed p-values for 332 pairs. Call filter_backbone() to apply thresholds.
Applying FDR (Benjamini-Hochberg) correction...
Found 0 significant edges (alpha=0.05, correction=fdr)


<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 0 stored elements and shape (34, 34)>

In [4]:
G_backbone = cmvp.to_networkx()
print(f"Original:  {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Backbone:  {G_backbone.number_of_nodes()} nodes, {G_backbone.number_of_edges()} edges")

Original:  34 nodes, 78 edges
Backbone:  34 nodes, 0 edges


## Re-filtering without recomputation

`filter_backbone()` only re-applies a threshold to the p-values already
computed by `validate_projection()` — call it again with a different
`alpha` (or `target_density`) to get a denser or sparser backbone instantly.

In [5]:
cmvp.filter_backbone(target_density=0.3)
G_dense = cmvp.to_networkx()
print(f"Denser backbone: {G_dense.number_of_edges()} edges")

Density-based thresholding: target=0.3000, actual=0.3012 (100/332 edges)
Denser backbone: 100 edges
